In [ ]:
from datetime import datetime
import shutil
import torch
import numpy as np
import rasterio
import os
from rasterio.windows import Window
from time import sleep

# DONE: Made test loader georeference LST
# DONE: Make test loader georeference all the rest to check ground truth against original as tile
# DONE: Test making each tile
# DONE: Test 0.0 overlap
# DONE: Organize to inference script and test the data loading more
# DONE: Make model prediction also pass geo reference and test against original as tile
# DONE: Add Heat index 1-25 into preprocess
# DONE: Make Heat Index show up in test quality
# DONE: Make it show in inference
# DONE: Train a preliminary model with 20 epochs, batch 1
# DONE: Test the model in inference
# DONE: Create a nowcast, 3 month cast and 6 month cast according to some parameter
# DONE: Get batching working -> max out the VRAM
# UNECESSARY: In inference, combine the tiles by city. Make it possible to do partial. Don't combine not in same city
# DONE: Make sure 512 and batching does have proper loss
# DONE: Profile the pipeline
# DONE: Get stats of all
# DONE: Preprocess all to float 32 with same nodata -9999 AND test
# DONE: Implement normalization
# DONE: Test data quality of data loader
# DONE: Fix denormalize
# DONE: Fix Tiled
# DONE: Gut and remove LandsatDataModule to TiledLandsatDataModule
# DONE: Preprocess all
# DONE: Add true normalization
# DONE: Create saving checkpoints by folder, test one 1 epoch
# DONE: Use Wandb with alert and Slack, setup phone
# DONE: Train and test with augmentation and batch sizing
# DONE: Test all 4 models for test loader accuracy, no artifact saved. Instead include a test loader test whenever val F is at its best. Make each on connect in the graph
# DONE: Test proper loss with normalization
# DONE: Fix labelling, metrics and inferencing. Test validate and train
# DONE: Test and correct inference
# DONE: Validate quality of 3 month and 6 month ahead data loaders
# DONE: Change all to 128
# DONE: Double check quality of 128. Do all three loaders then all three loaders again with augmentation. Use like 45 tiles.
# DONE: Do a full debug run to check underfitting
# TODO: Compare to the good originals
# TODO: Check with Isaac's recommendations
# DONE: Double check normalization, tile size
# TODO: Debug underfitting in debug
# TODO: Try one FULL data run on the 4 GPUs
# TODO: Debug full run
# TODO: Fix mixed precision
# TODO: Test debug run on mixed precision
# TODO: Test full run on mixed precision
# TODO: Tune batch size
# TODO: Train one good model on 3 month prediction Batch 32, Augment True, debug true
# TODO: Compare 3 month accuracy to 0 month
# DONE: Migrate to Jetstream
# DONE: Create a way to look 0, 3, 6 months ahead
# TODO: Test by zero-shot, city only, temporal only, both, debug true, Use the table
# TODO: Debug False, Test by zero-shot, city only, temporal only, both, Use the table
# TODO: Report to Isaac and show table
#*Merging in inference
#*Maybe use open street map
#*Maybe use NDBI


In [2]:
from utils.data.TiledLandsatDataModule import TiledGeotiffDataset

def inference(model, test_loader, tiles_count: int, device='cuda', denormalize: bool=False):
    model = model.to(device)
    model.eval()    
    batch = 0
    it = iter(test_loader)
    with torch.no_grad():
        for _ in range(tiles_count):
            sleep(1)
            # Get one sample
            sample = next(it)
            for l, outTif in enumerate(['LST.tif', 'HeatIndex.tif']):
                inputs = sample['input'].to(device)
                targets = sample['target'].to(device)
                mask = sample['mask'].to(device)
                ground_truth_file = sample['file_dict'][outTif][0]
                box = sample['box']
                box = [int(tensor.item()) for tensor in box]

                # Get model prediction
                outputs = model(inputs)
                if denormalize:
                    print(outputs.shape)
                    outputs = TiledGeotiffDataset.denormalize(outputs)[batch][l]
                    targets = TiledGeotiffDataset.denormalize(targets)[batch][l]

                # Move to CPU and convert to numpy
                mask_np = mask.cpu().numpy().squeeze()
                targets_np = targets.cpu().numpy().squeeze()
                predicted_np = outputs.cpu().numpy().squeeze()
                
                # Apply mask
                predicted_np[~mask_np] = -9999
                targets_np[~mask_np] = -9999

                output_dir = "./Data/prediction"
                os.makedirs(output_dir, exist_ok=True)
                xmin, ymin, xmax, ymax = box
                window = Window(col_off=xmin, row_off=ymin, width=xmax-xmin, height=ymax-ymin)
                
                # Get corresponding LST file path and save outputs
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                with rasterio.open(ground_truth_file) as src:
                    profile = src.profile.copy()
                    # Update the transform based on the window
                    window_transform = rasterio.windows.transform(window, src.transform)
                    
                    # Update profile with new dimensions and transform
                    profile.update(
                        width=xmax-xmin,
                        height=ymax-ymin,
                        transform=window_transform,
                        count=1,
                        nodata=-9999
                    )
                    
                    # Save prediction
                    pred_filename = os.path.join(output_dir, f'predicted_{timestamp}_{outTif}')
                    with rasterio.open(pred_filename, "w", **profile) as dst:
                        dst.write(predicted_np.astype(np.float32), 1)
                    
                    # Save ground truth
                    truth_filename = os.path.join(output_dir, f'ground_truth_{outTif}_{timestamp}.tif')
                    with rasterio.open(truth_filename, "w", **profile) as dst:
                        dst.write(targets_np.astype(np.float32), 1)
                    
                    # Copy original out file
                    orig_filename = os.path.join(output_dir, f'original_{outTif}')
                    if not os.path.exists(orig_filename):
                        print(f"Original LST: {os.path.basename(orig_filename)}")
                        shutil.copy2(ground_truth_file, orig_filename)
                    
                    # Calculate metrics for valid pixels
                    valid_mask = predicted_np != -9999
                    if valid_mask.any():
                        mae = np.mean(np.abs(predicted_np[valid_mask] - targets_np[valid_mask]))
                        rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
                        metrics = {'mae': mae, 'rmse': rmse}
                        print(f"Predictions: {os.path.basename(pred_filename)}")
                        # print(f"Ground Truth: {os.path.basename(truth_filename)}")
                        if 'Heat' in pred_filename:
                            print(f"Mean Absolute Error: {mae:.2f} points.")
                            print(f"Root Mean Square Error: {rmse:.2f} points.")
                        else:
                            print(f"Mean Absolute Error: {mae:.2f}°F")
                            print(f"Root Mean Square Error: {rmse:.2f}°F")
                print(f"\nSaved files in {output_dir}/:")
        
    # return metrics

def test_data_quality(test_loader, tiles_count: int, denormalize: bool = False):
    batch = 0
    it = iter(test_loader)
    for _ in range(tiles_count):
        sleep(1)
        # Get one sample
        sample = next(it)
        if denormalize:
            sample = TiledGeotiffDataset.denormalize(sample)
        for i, tif in enumerate(['Albedo.tif', 'DEM.tif', 'Land_Cover.tif', 'NDVI.tif', 'NDWI.tif', 'LST.tif', 'HeatIndex.tif']):
            with torch.no_grad():
                if i <= 4:
                    targets = sample['input'][batch][i]
                else:
                    targets = sample['target'][batch][i-5]
                mask = sample['mask']
                target_file_origin = sample['file_dict'][tif][0]
                box = sample['box']
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                box = [int(tensor.item()) for tensor in box]
                # print(box)

                # Move to CPU and convert to numpy
                mask_np = mask.cpu().numpy().squeeze()
                targets_np = targets.cpu().numpy().squeeze()

                noData = -9999
                targets_np[~mask_np] = noData
                # Save ground truth
                output_dir = "./Data/truth"
                os.makedirs(output_dir, exist_ok=True)
                xmin, ymin, xmax, ymax = box
                window = Window(col_off=xmin, row_off=ymin, width=xmax-xmin, height=ymax-ymin)
                
                print(target_file_origin)
                with rasterio.open(target_file_origin) as src:
                    # Get the profile from the source
                    test_profile = src.profile.copy()
                    
                    # Update the transform based on the window
                    window_transform = rasterio.windows.transform(window, src.transform)
                    
                    # Update profile with new dimensions and transform
                    test_profile.update(
                        width=xmax-xmin,
                        height=ymax-ymin,
                        transform=window_transform,
                        count=1,
                        nodata=noData
                    )
                    
                    truth_filename = os.path.join(output_dir, f'ground_truth_{timestamp}_{tif}')
                    with rasterio.open(truth_filename, "w", **test_profile) as dst:
                        dst.write(targets_np.astype(np.float32), 1)

                # orig_filename = os.path.join(output_dir, f'original_File_{timestamp}_{tif}')
                # with rasterio.open(target_file_origin) as src:
                #     profile = src.profile.copy()
                #     profile.update(dtype='float32', count=1, nodata=np.nan)
                #     print(f"\nSaved files in {output_dir}/:")
                #     print(f"Ground Truth: {os.path.basename(truth_filename)}")
                #     print(f"Original LST: {os.path.basename(orig_filename)}")
                #     # Copy original output files
                #     print(f'Copying {target_file_origin} to {orig_filename}')
                #     if not os.path.exists(orig_filename):
                #         shutil.copy2(target_file_origin, orig_filename)

In [3]:
# import wandb
# import os

# os.environ["WANDB_NOTEBOOK_NAME"] = "TrainUNet-Basic.ipynb"
# os.environ["WANDB_DIR"] = "./wandb"
# os.environ["WANDB_CACHE_DIR"] = "/work/ubh496/.cache/wandb"
# os.environ["WANDB_CONFIG_DIR"] = "/work/ubh496/.config/wandb"
# os.environ["WANDB_DATA_DIR"] = "/work/ubh496/.cache/wandb-data"
# os.environ["WANDB_ARTIFACT_DIR"] = "./artifacts"

# run = wandb.init(dir="/work/ubh496/heat-island-test/wandb/downloaded_models")
# artifact = run.use_artifact('jesus-guerrero-ml/heat-island/model-sclb910d:v8', type='model')
# artifact_dir = artifact.download()

In [ ]:
from utils.data.TiledLandsatDataModule import TiledLandsatDataModule
from utils.model import LSTNowcaster

config = {
    "debug": True,
    "augment": False,
    "by_city": False,
    "tile_size": 128,
    "tile_overlap": 0.0,
    "months_ahead": 0,
    "learning_rate": 1e-4,
    "model": "unet",
    "backbone": "resnet50",
    "dataset": "pure_landsat",
    "epochs": 25,
    "batch_size": 1,
    "pretrained_weights": True,
    "deterministic": True,
    "in_channels": 5
}

best_model = LSTNowcaster()

best_model = LSTNowcaster.load_from_checkpoint(
    "/home/ubuntu/heat-island-test/wandb/heat-island/checkpoints/sgfs7yoi_March19/sgfs7yoi_March19_epoch=264_val_rmse_F=13.4460.ckpt"
)

data_module = TiledLandsatDataModule(
    data_dir="./Data",
    monthsAhead=config["months_ahead"],
    augment=config["augment"],
    shuffleTrain=False,
    batch_size=1,
    num_workers=5,        
    tile_size=config["tile_size"],
    includeYears=["2018", "2019", "2020"]
)
# Setup the data module to prepare datasets
data_module.setup()
print(f'Test is length {len(data_module.test_dataloader())}')
print(f'Validate is length {len(data_module.val_dataloader())}')
print(f'Train is length {len(data_module.train_dataloader())}')


Preparing scene by scene...: 100%|██████████| 1954/1954 [00:00<00:00, 10218.58it/s]


Dataset splits - Train: 924, Val: 116, Test: 116
Test is length 8592
Validate is length 8400
Train is length 68304


In [5]:
test_data_quality(
    test_loader=data_module.train_dataloader(), tiles_count=500, denormalize=True
)

# inference(
#     model=best_model, test_loader=data_module.train_dataloader(), tiles_count=3, denormalize=True
# )


/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/Albedo.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/DEM.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/Land_Cover.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/NDVI.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/NDWI.tif
/home/ubuntu/heat-island-test/Data/preprocess/y/less5CloudCover/Buckeye_AZ/2018-11/LST.tif
/home/ubuntu/heat-island-test/Data/preprocess/y/less5CloudCover/Buckeye_AZ/2018-11/HeatIndex.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/Albedo.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/DEM.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCover/Buckeye_AZ/2018-08/Land_Cover.tif
/home/ubuntu/heat-island-test/Data/preprocess/X/less5CloudCove